# 07.4 - Text Classification

**Phase:** 07 - NLP

**Status:** VERIFIED

---

## 1. What Are We Solving?

Assigning a label (spam/ham, positive/negative, topic) to a piece of text. This unit chains preprocessing -> vectorization -> classifier into one complete ML pipeline.

## 2. Why Does This Matter?

Text classification powers spam filters, sentiment analysis, topic labeling, intent detection, and content moderation. It is one of the most deployed NLP applications, and it's where classical ML (fast, interpretable) shines.

## 3. Prerequisites

- Units 07.1-07.3 (preprocessing, BoW/TF-IDF, N-grams)
- Phase 05 (ML classifiers)

## 4. Learning Objectives

By the end of this unit, you should be able to:
- Build a scikit-learn Pipeline (vectorizer + classifier)
- Compare Naive Bayes, Logistic Regression, and SVM on text
- Evaluate with precision/recall/F1 and a confusion matrix
- Handle class imbalance and avoid data leakage

## 5. Mental Model

```text
Raw text -> preprocess -> vectorize (TF-IDF) -> classifier -> label
```
The Pipeline object guarantees the vectorizer is fit on train only, preventing data leakage.


## 6. Setup + Data

We build a small binary sentiment corpus.


In [1]:
import matplotlib
matplotlib.use('Agg')
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix, f1_score

texts = [
    "I love this product it is fantastic",
    "Terrible experience waste of money",
    "Great value highly recommend",
    "Very disappointing quality",
    "Amazing quality and fast shipping",
    "Not worth it avoid this",
    "Best purchase I ever made",
    "Poor customer service never again",
    "Wonderful and delightful, love it",
    "Horrible and broken, do not buy",
]
labels = [1, 0, 1, 0, 1, 0, 1, 0, 1, 0]  # 1 = positive

X_tr, X_te, y_tr, y_te = train_test_split(texts, labels, test_size=0.3, random_state=42, stratify=labels)
print(f"Train: {len(X_tr)}, Test: {len(X_te)}")


Train: 7, Test: 3


## 7. Build a Pipeline (No Data Leakage)


In [2]:
pipe = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2), max_features=5000)),
    ('clf', LogisticRegression(max_iter=1000))
])

pipe.fit(X_tr, y_tr)
preds = pipe.predict(X_te)
print("Predictions:", preds)
print("True labels :", y_te)
print("\nF1:", round(f1_score(y_te, preds), 3))


Predictions: [1 1 1]
True labels : [0, 0, 1]

F1: 0.5


## 8. Compare Classifiers

Logistic Regression, Naive Bayes, and Linear SVM are the classic text classifiers.


In [3]:
classifiers = {
    'LogisticRegression': LogisticRegression(max_iter=1000),
    'MultinomialNB': MultinomialNB(),
    'LinearSVC': LinearSVC(max_iter=10000),
}

for name, clf in classifiers.items():
    p = Pipeline([('tfidf', TfidfVectorizer(ngram_range=(1, 2))), ('clf', clf)])
    scores = cross_val_score(p, texts, labels, cv=3, scoring='f1_macro')
    print(f"{name:20s} F1-macro (CV): {scores.mean():.3f}")


LogisticRegression   F1-macro (CV): 0.278


MultinomialNB        F1-macro (CV): 0.250


LinearSVC            F1-macro (CV): 0.194


## 9. Interpret Logistic Regression Coefficients

For text classification, LR weights tell you which words push positive/negative.


In [4]:
clf = pipe.named_steps['clf']
feats = pipe.named_steps['tfidf'].get_feature_names_out()
coef = clf.coef_[0]
top_pos = np.argsort(coef)[-5:]
top_neg = np.argsort(coef)[:5]
print("Most positive words:", [feats[i] for i in top_pos])
print("Most negative words:", [feats[i] for i in top_neg])


Most positive words: ['purchase', 'purchase ever', 'made', 'value', 'value highly']
Most negative words: ['not', 'disappointing', 'very disappointing', 'disappointing quality', 'very']


## 10. Multinomial Naive Bayes from a Hand Corpus


In [5]:
m = MultinomialNB()
tfidf = TfidfVectorizer(stop_words='english')
Xm = tfidf.fit_transform(X_tr)
m.fit(Xm, y_tr)
print("NB test acc:", round(m.score(tfidf.transform(X_te), y_te), 3))
print("predict('free prize waiting for you'):", m.predict(tfidf.transform(["free prize waiting for you"]))[0])


NB test acc: 0.333
predict('free prize waiting for you'): 1


## 11. Failure Case: Class Imbalance

Accuracy is misleading when one class dominates. Build an imbalanced dataset.


In [6]:
imb_texts = ["ok fine product", "fine okay"] * 5 + ["awful terrible horrible worst ever"]
imb_y = [1]*10 + [0]  # 10 positives, 1 negative

p = Pipeline([('tfidf', TfidfVectorizer()), ('clf', LogisticRegression(max_iter=1000))])
p.fit(imb_texts, imb_y)
print("Predictions on all docs:", p.predict(imb_texts))
print("Accuracy on train:", round(p.score(imb_texts, imb_y), 3), "(looks great?)")
print("But the minority class is never predicted -> F1 would be 0.")
print("Lesson: report F1 / per-class metrics, not accuracy alone.")


Predictions on all docs: [1 1 1 1 1 1 1 1 1 1 1]
Accuracy on train: 0.909 (looks great?)
But the minority class is never predicted -> F1 would be 0.
Lesson: report F1 / per-class metrics, not accuracy alone.


## 12. Handle Imbalance with class_weight


In [7]:
p2 = Pipeline([('tfidf', TfidfVectorizer()),
               ('clf', LogisticRegression(max_iter=1000, class_weight='balanced'))])
p2.fit(imb_texts, imb_y)
print("Balanced model predicts:", p2.predict(imb_texts)[-3:])
print("class_weight='balanced' downweights the majority class.")


Balanced model predicts: [1 1 0]
class_weight='balanced' downweights the majority class.


## 13. Confusion Matrix

See where the model confuses classes.


In [8]:
cm = confusion_matrix(y_te, pipe.predict(X_te))
print("Confusion matrix (rows=actual, cols=predicted):")
print(cm)
print("\nDiagonal = correct; off-diagonal = errors.")


Confusion matrix (rows=actual, cols=predicted):
[[0 2]
 [0 1]]

Diagonal = correct; off-diagonal = errors.


## 14. Debugging: Common Errors

- **Always predicts majority class** — imbalance. Fix: `class_weight='balanced'` or resample.
- **Perfect train, poor test** — overfitting. Fix: fewer features, more regularization.
- **Convergence warning** — not enough iterations. Fix: raise `max_iter`.
- **Poor on short texts** — too few features. Fix: char n-grams.

## 15. Real-World Considerations

- Always start with Logistic Regression / Naive Bayes baselines.
- Use `classification_report` over accuracy alone.
- Tune vectorizer and classifier as a joint pipeline.

## 16. Common Mistakes

- Fitting vectorizer before splitting (leakage).
- Using accuracy on imbalanced data.
- Jumping to neural networks before trying a simple baseline.

## 17. When NOT to Use

- Sparse high-dim + non-linear boundaries (maybe Random Forest, but check).
- Large datasets where neural models clearly win (Phase 08).

## 18. Challenge

Add `class_weight='balanced'` to the main pipeline and compare F1 vs the default.


In [9]:
# Challenge: compare default vs balanced classifier
def f1pipe(clf):
    p = Pipeline([('tfidf', TfidfVectorizer(ngram_range=(1, 2))), ('clf', clf)])
    return cross_val_score(p, texts, labels, cv=3, scoring='f1_macro').mean()

print("Default LR F1:", round(f1pipe(LogisticRegression(max_iter=1000)), 3))
print("Balanced LR F1:", round(f1pipe(LogisticRegression(max_iter=1000, class_weight='balanced')), 3))


Default LR F1:

 0.278


Balanced LR F1:

 0.194


## 19. Closed-Book Recall

1. Why does a Pipeline prevent data leakage?
2. Name three good text classifiers and when you'd pick each.
3. Why is accuracy misleading on imbalanced data?
4. What does `class_weight='balanced'` do?

## 20. Teach-Back Questions

- Walk through building a spam classifier end to end.
- Explain precision, recall, and F1 with a spam example.

## 21. Summary

You built complete text classification pipelines, compared three classifiers, interpreted coefficients, and handled class imbalance. This is a reusable template for any text tagging task.

## 22. Further Experiment

- Use `fetch_20newsgroups` for a multi-class topic classifier.
- Tune `max_features` and `ngram_range` jointly with GridSearchCV.

## 23. Verification Status

```
STATUS: VERIFIED
EXECUTION: PASS
DEPENDENCIES: numpy, scikit-learn, matplotlib
OUTPUTS: PASS
LAST VERIFIED: 2026-08-29
```
